# IWAE: 複数の潜在候補で下界をきつくする

IWAEは、VAEの潜在サンプルをK個に増やし、importance weightを使ってよりきつい下界を作る。


## このノートの読み方

想定読者: VAE、ELBO、log probability、PyTorchのbroadcastを理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

VAEは1つのzでELBOを評価した。IWAEはK個のz候補を使い、説明力の高い候補をlogsumexpで安定にまとめる。


## 到達目標

- IWAE boundを説明できる
- `K x batch x z_dim`のshapeを追える
- `logsumexp`の必要性を説明できる


## 重要語句

- `importance weight`: qからサンプルしたzをp/qで補正する重み
- `K samples`: 1つのxに対する複数の潜在候補
- `logsumexp`: log空間で和を安定に計算する操作


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| z | (K, B, z_dim) | K個の潜在候補 |
| log_w | (K, B) | 重要度重み |
| bound | (B,) | サンプルごとの下界 |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| VAEとの関係 | `K=1`のIWAE boundはVAEのELBOに対応する。Kを増やすと候補zを複数見られる。 |
| 期待値の対象 | `z_{1:K}`は`q_phi(z|x)`から独立にサンプルする。boundの単調性は期待値としての性質で、単一乱数では揺れる。 |
| log weight | `log_w = log p(x|z)+log p(z)-log q(z|x)`で、コードの各項と1対1に対応する。 |
| 安定化 | `logsumexp(log_w)-log(K)`までがIWAE boundの安定計算である。expして平均すると桁落ちしやすい。 |
| バイオ用途 | 単一細胞の潜在表現のように観測xを説明するz候補が複数あり得る場合、K候補で下界をきつく見る発想につながる。 |


## IWAE bound

K個の重みの平均をlogで評価する。

$$
\mathcal{L}_K=\mathbb{E}\left[\log\frac{1}{K}\sum_{k=1}^K w_k\right]
$$


## Importance weight

qから出やすいだけのzを補正し、モデル上の説明力を見る。

$$
w_k=\frac{p_\theta(x,z_k)}{q_\phi(z_k|x)}
$$


## Stable implementation

直接expしてmeanすると数値不安定になりやすい。

$$
\log\sum_k\exp(a_k)=m+\log\sum_k\exp(a_k-m)
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
K, batch = 5, 3
log_px_z = torch.randn(K, batch) - 1.0
log_pz = torch.randn(K, batch) - 0.5
log_qz_x = torch.randn(K, batch) - 0.2
log_w = log_px_z + log_pz - log_qz_x
bound = torch.logsumexp(log_w, dim=0) - math.log(K)
print("log_w shape:", log_w.shape)
print("IWAE bound:", bound.round(decimals=3))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/iwae_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/iwae_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/iwae_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="IWAE: 複数の潜在候補で下界をきつくする difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### K samples animation

- 学習目標: 1つのxからK個のzを作る
- 誤解の防止: batchとKを混同する

対応する式:

$$
z_{1:K}\sim q_\phi(z|x)
$$


<p><a href="../demos/iwae_k_samples.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/iwae_k_samples.html</code>）</p>
<iframe
  src="../demos/iwae_k_samples.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="K samples animation"
></iframe>


### importance weights animation

- 学習目標: 候補ごとのlog_wを棒で見る
- 誤解の防止: 重みが確率そのものだと思う

対応する式:

$$
\log w=\log p(x,z)-\log q(z|x)
$$


<p><a href="../demos/iwae_weights.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/iwae_weights.html</code>）</p>
<iframe
  src="../demos/iwae_weights.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="importance weights animation"
></iframe>


### K comparison animation

- 学習目標: K=1とK>1を比べる
- 誤解の防止: Kを増やせば常に楽だと思う

対応する式:

$$
\mathcal{L}_1\le\mathcal{L}_K\le\log p(x)
$$


<p><a href="../demos/iwae_k_compare.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/iwae_k_compare.html</code>）</p>
<iframe
  src="../demos/iwae_k_compare.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="K comparison animation"
></iframe>


### stable logsumexp animation

- 学習目標: maxを引いて安定に足す
- 誤解の防止: expして平均すれば十分と思う

対応する式:

$$
m+\log\sum\exp(a_k-m)
$$


<p><a href="../demos/iwae_logsumexp.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/iwae_logsumexp.html</code>）</p>
<iframe
  src="../demos/iwae_logsumexp.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="stable logsumexp animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`IWAE: 複数の潜在候補で下界をきつくする`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyIWAEDataset(Dataset):
    def __init__(self, n_samples: int = 32, x_dim: int = 4) -> None:
        self.features = torch.randn(n_samples, x_dim)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"features": self.features[index]}


class TrainerIWAE(nn.Module):
    def __init__(self, x_dim: int = 4, z_dim: int = 2, k_samples: int = 5) -> None:
        super().__init__()
        self.k_samples = k_samples
        self.encoder = nn.Sequential(nn.Linear(x_dim, 16), nn.ReLU())
        self.mu = nn.Linear(16, z_dim)
        self.logvar = nn.Linear(16, z_dim)
        self.decoder = nn.Sequential(nn.Linear(z_dim, 16), nn.ReLU(), nn.Linear(16, x_dim))

    def forward(self, features: torch.Tensor) -> dict[str, torch.Tensor]:
        h = self.encoder(features)
        mu = self.mu(h)
        logvar = self.logvar(h)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn(self.k_samples, *mu.shape, device=features.device)
        z = mu.unsqueeze(0) + std.unsqueeze(0) * eps
        recon_mu = self.decoder(z)
        x = features.unsqueeze(0)
        log_px_z = -0.5 * torch.sum((x - recon_mu) ** 2 + math.log(2 * math.pi), dim=-1)
        log_pz = -0.5 * torch.sum(z ** 2 + math.log(2 * math.pi), dim=-1)
        log_qz_x = -0.5 * torch.sum(((z - mu.unsqueeze(0)) / std.unsqueeze(0)) ** 2 + logvar.unsqueeze(0) + math.log(2 * math.pi), dim=-1)
        log_w = log_px_z + log_pz - log_qz_x
        bound = torch.logsumexp(log_w, dim=0) - math.log(self.k_samples)
        loss = -torch.mean(bound)
        return {"loss": loss, "logits": recon_mu[0]}


training_args = TrainingArguments(
    output_dir="./results/iwae_trainer_demo",
    max_steps=1,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

trainer = Trainer(model=TrainerIWAE(), args=training_args, train_dataset=TinyIWAEDataset())
train_output = trainer.train()
print("IWAE Trainer loss:", train_output.training_loss)


batch = next(iter(torch.utils.data.DataLoader(TinyIWAEDataset(n_samples=8), batch_size=4)))
for k in [1, 5, 20]:
    trainer.model.k_samples = k
    with torch.no_grad():
        estimate = -trainer.model(batch["features"])["loss"]
    print(f"K={k} bound estimate:", float(estimate.detach()))

with torch.no_grad():
    z_prior = torch.randn(3, 2)
    generated = trainer.model.decoder(z_prior)
print("IWAE prior sample shape:", generated.shape)


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- K=1,5,20の推定を比較する
- log_wの各項を表示する
- importance samplingの分散を観察する


## 確認問題

- `K=1`のIWAE boundは何に対応するか。
- `logsumexp`を使う理由を書く。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
